# Connectome-GAN
**Transforming resting-state brain networks into task-specific functional connectomes**

This notebook implements a pix2pix-style GAN that takes a subject's **resting-state functional
connectome (FC)** as input and generates the corresponding **task FC**. Connectivity matrices are
built from the **268-node Shen atlas**, so every matrix is `268 × 268`.

Pipeline: **data loading → loss functions → generator / discriminator → training (leave-one-out CV)**.

> The network operates directly on `268 × 268` matrices. The encoder is downsampled twice with
> stride-2 convolutions (`268 → 134 → 67`) and the decoder mirrors it (`67 → 134 → 268`).
> No zero-padding is used: the architecture bottoms out naturally at a spatial size of **67**.

In [ ]:
import os, time, datetime, pathlib

import h5py
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
tf.keras.backend.set_floatx('float32')

print("TensorFlow:", tf.__version__)

## Configuration

In [ ]:
N_NODES    = 268        # Shen 268-node atlas -> 268 x 268 connectivity matrix
IMG_SIZE   = N_NODES    # network input/output spatial size
CHANNELS   = 1
N_SUBJECTS = 316        # number of HCP subjects

## 1. Data loading

HCP connectomes are stored in MATLAB v7.3 `.mat` files (HDF5 format), so we read them with `h5py`.
We load the **resting-state FC** (model input) and the **task FC** (target / output), enforce
matrix symmetry, and add a channel axis to obtain tensors of shape `(N_subjects, 268, 268, 1)`.

> Connectivity matrices are symmetric by construction. We apply `0.5 * (A + Aᵀ)` as a safeguard so
> the targets match the symmetric output produced by the generator (see the `sym` layer below).

In [ ]:
# Paths to the HCP connectome data
CONMAT_PATH  = '/home/nas/HCP_316/conmat_HCP_run_316_201216.mat'
SUBINFO_PATH = '/home/nas/HCP_316/subInfo_316_attention_210226.mat'

# Inspect the variable names stored in the file (run once, then set the keys below)
with h5py.File(CONMAT_PATH, 'r') as f:
    print("Variables in file:", list(f.keys()))

In [ ]:
def load_connectomes(mat_path, var_name):
    """Load a (N_subjects, 268, 268) connectome stack from a MATLAB v7.3 (.mat) file."""
    with h5py.File(mat_path, 'r') as f:
        data = np.array(f[var_name], dtype=np.float32)
    # h5py returns MATLAB arrays with reversed axis order; restore (subjects, nodes, nodes)
    if data.ndim == 3 and data.shape[0] != N_SUBJECTS:
        data = np.transpose(data, (2, 1, 0))
    return np.round(data, 3)

def symmetrize(mat):
    """Enforce symmetry: 0.5 * (A + A^T), applied to the last two axes."""
    return 0.5 * (mat + np.swapaxes(mat, -1, -2))

In [ ]:
# >>> Replace 'conMatRun268' / 'conMatTask268' with the actual keys in your .mat file <<<
rest_fc = load_connectomes(CONMAT_PATH, 'conMatRun268')    # resting-state FC  (input)
task_fc = load_connectomes(CONMAT_PATH, 'conMatTask268')   # task FC           (target)

rest_fc = symmetrize(rest_fc)
task_fc = symmetrize(task_fc)

# add channel dimension -> (N, 268, 268, 1)
input_data  = rest_fc[..., np.newaxis].astype('float32')
output_data = task_fc[..., np.newaxis].astype('float32')

print("input  (rest):", input_data.shape)
print("output (task):", output_data.shape)

## 2. Structure-preserving loss

In addition to the adversarial loss and an L1 reconstruction term, we add a
**structure-preserving loss** based on the Pearson correlation coefficient (PCC) computed along
the **rows** and **columns** of the connectome. This encourages the generated matrix to keep the
same row/column connectivity profile as the empirical task FC, preserving network topology.

`L_total = L_GAN + λ · ( L1 + L_structure-preserving )`

In [ ]:
def pearson_correlation_coefficient_loss_brc(y_true, y_pred, epsilon=1e-6):
    """Structure-preserving loss: row-wise + column-wise (1 - PCC).
    Input tensors have shape (batch, H, W, channel)."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # --- Row-wise PCC (correlate each row across columns) ---
    mt_r = tf.reduce_mean(y_true, axis=[2, 3], keepdims=True)
    mp_r = tf.reduce_mean(y_pred, axis=[2, 3], keepdims=True)
    ct_r, cp_r = y_true - mt_r, y_pred - mp_r
    cov_r = tf.reduce_sum(ct_r * cp_r, axis=[2, 3])
    sd_t_r = tf.sqrt(tf.reduce_sum(tf.square(ct_r), axis=[2, 3]))
    sd_p_r = tf.sqrt(tf.reduce_sum(tf.square(cp_r), axis=[2, 3]))
    pcc_r = cov_r / (sd_t_r * sd_p_r + epsilon)
    loss_r = tf.reduce_sum(1.0 - pcc_r, axis=1)

    # --- Column-wise PCC (correlate each column across rows) ---
    mt_c = tf.reduce_mean(y_true, axis=[1, 3], keepdims=True)
    mp_c = tf.reduce_mean(y_pred, axis=[1, 3], keepdims=True)
    ct_c, cp_c = y_true - mt_c, y_pred - mp_c
    cov_c = tf.reduce_sum(ct_c * cp_c, axis=[1, 3])
    sd_t_c = tf.sqrt(tf.reduce_sum(tf.square(ct_c), axis=[1, 3]))
    sd_p_c = tf.sqrt(tf.reduce_sum(tf.square(cp_c), axis=[1, 3]))
    pcc_c = cov_c / (sd_t_c * sd_p_c + epsilon)
    loss_c = tf.reduce_sum(1.0 - pcc_c, axis=1)

    return tf.reduce_mean(loss_r + loss_c)

## 3. Building blocks

Reusable down-/up-sampling blocks. Down-sampling uses strided `Conv2D` + LeakyReLU;
up-sampling uses strided `Conv2DTranspose` + ReLU, with optional spatial dropout.

In [ ]:
def downsample(filters, size, strides, apply_dropout=False, name=None):
    init = tf.random_normal_initializer(0., 0.02)
    block = tf.keras.Sequential(name=name)
    block.add(tf.keras.layers.Conv2D(filters, size, strides=strides, padding='same',
                                     kernel_initializer=init, use_bias=False))
    if apply_dropout:
        block.add(tf.keras.layers.SpatialDropout2D(0.3))
    block.add(tf.keras.layers.LeakyReLU(alpha=0.1))
    return block

def upsample(filters, size, strides, apply_dropout=False, name=None):
    init = tf.random_normal_initializer(0., 0.02)
    block = tf.keras.Sequential(name=name)
    block.add(tf.keras.layers.Conv2DTranspose(filters, size, strides=strides, padding='same',
                                              kernel_initializer=init, use_bias=False))
    if apply_dropout:
        block.add(tf.keras.layers.SpatialDropout2D(0.3))
    block.add(tf.keras.layers.ReLU())
    return block

## 4. Generator (U-Net)

A compact U-Net operating on `268 × 268` matrices:

| stage | op | spatial size |
|-------|-----|--------------|
| input | — | 268 |
| down1 | stride-2 conv | 134 |
| down2 | stride-2 conv | **67** |
| bottleneck | stride-1 conv | 67 |
| up1 | stride-2 transpose conv (+ skip from down1) | 134 |
| output | stride-2 transpose conv | 268 |

The final `sym` layer symmetrizes the output, `0.5 · (A + Aᵀ)`, because functional connectomes are
symmetric. No padding is applied anywhere; the encoder stops exactly at a spatial size of 67.

In [ ]:
def Generator():
    inputs = tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS), dtype=tf.float32)
    init = tf.random_normal_initializer(0., 0.02)

    # ---- Encoder ----
    d1 = downsample(128, 4, strides=2, name='down1')(inputs)   # 268 -> 134
    d2 = downsample(256, 4, strides=2, name='down2')(d1)       # 134 -> 67

    # ---- Bottleneck (stride 1, keeps 67) ----
    b  = downsample(256, 4, strides=1, name='bottleneck')(d2)  # 67 -> 67

    # ---- Decoder + skip connection ----
    u1  = upsample(128, 4, strides=2, apply_dropout=True, name='up1')(b)  # 67 -> 134
    u1c = tf.keras.layers.Concatenate()([u1, d1])                          # concat with down1 (134)

    out_raw = tf.keras.layers.Conv2DTranspose(
        CHANNELS, 4, strides=2, padding='same',
        kernel_initializer=init, activation='linear', name='out_raw')(u1c)  # 134 -> 268

    # ---- Symmetrize: 0.5 * (A + A^T) ----
    out_t = tf.transpose(out_raw, perm=[0, 2, 1, 3])
    out = tf.keras.layers.Lambda(lambda x: 0.5 * (x[0] + x[1]), name='sym')([out_raw, out_t])

    return tf.keras.Model(inputs=inputs, outputs=out, name='Generator')

## 5. Discriminator (PatchGAN)

A conditional PatchGAN that receives the resting-state FC and a task FC (real or generated)
concatenated along the channel axis, and outputs a `67 × 67` map of real/fake logits. It is also
kept "to 67 only" — two stride-2 layers, then stride-1 layers that preserve resolution.

In [ ]:
def Discriminator():
    init = tf.random_normal_initializer(0., 0.02)
    inp = tf.keras.layers.Input(shape=[IMG_SIZE, IMG_SIZE, CHANNELS], name='input_image')
    tar = tf.keras.layers.Input(shape=[IMG_SIZE, IMG_SIZE, CHANNELS], name='target_image')

    x = tf.keras.layers.concatenate([inp, tar])     # (batch, 268, 268, 2)

    d0 = downsample(128, 4, strides=2)(x)            # 268 -> 134
    d1 = downsample(256, 4, strides=2)(d0)           # 134 -> 67
    d2 = downsample(512, 4, strides=1)(d1)           # 67  -> 67

    last = tf.keras.layers.Conv2D(1, 4, strides=1, padding='same',
                                  kernel_initializer=init)(d2)   # 67 -> 67 patch logits
    return tf.keras.Model(inputs=[inp, tar], outputs=last, name='Discriminator')

## 6. Losses and optimizers

The discriminator outputs raw logits, so we use `BinaryCrossentropy(from_logits=True)`
(standard pix2pix setup). The generator loss combines the adversarial term with an L1 term and the
structure-preserving PCC loss.

In [ ]:
LAMBDA = 1
loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(disc_generated_output, gen_output, target):
    gan_loss = loss_object(tf.ones_like(disc_generated_output), disc_generated_output)
    l1_loss  = tf.reduce_mean(tf.abs(target - gen_output))
    struct   = pearson_correlation_coefficient_loss_brc(target, gen_output)
    recon    = 1.0 * l1_loss + 0.5 * struct                 # L1 + structure-preserving
    total_gen_loss = gan_loss + LAMBDA * recon
    return total_gen_loss, gan_loss, recon

def discriminator_loss(disc_real_output, disc_generated_output):
    real_loss      = loss_object(tf.ones_like(disc_real_output),       disc_real_output)
    generated_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)
    return real_loss + generated_loss

generator_optimizer     = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5, beta_2=0.999)
discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5, beta_1=0.5, beta_2=0.999)

## 7. Evaluation metrics

In [ ]:
def spatial_correlation_corr2(y_true, y_pred):
    """2-D Pearson correlation between two matrices (equivalent to MATLAB corr2)."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    a = y_true - tf.reduce_mean(y_true)
    b = y_pred - tf.reduce_mean(y_pred)
    return tf.reduce_sum(a * b) / (tf.sqrt(tf.reduce_sum(a*a) * tf.reduce_sum(b*b)) + 1e-8)

def calculate_mse(predicted, actual):
    """Mean squared error between two NumPy arrays of equal shape."""
    predicted, actual = np.asarray(predicted), np.asarray(actual)
    if predicted.shape != actual.shape:
        raise ValueError(f"shape mismatch: {predicted.shape} vs {actual.shape}")
    return float(np.mean(np.square(predicted - actual)))

## 8. Build the models

In [ ]:
generator     = Generator()
discriminator = Discriminator()

generator.summary()
# tf.keras.utils.plot_model(generator, show_shapes=True, dpi=64)   # optional diagram

## 9. Training step

A single optimization step: generate a fake task FC, score real and fake pairs with the
discriminator, then update both networks. Decorated with `@tf.function` for speed.

In [ ]:
@tf.function(reduce_retracing=True)
def train_step(input_image, target):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator(input_image, training=True)

        disc_real = discriminator([input_image, target],     training=True)
        disc_gen  = discriminator([input_image, gen_output], training=True)

        gen_total_loss, gen_gan_loss, gen_l1_loss = generator_loss(disc_gen, gen_output, target)
        disc_loss = discriminator_loss(disc_real, disc_gen)

    gen_grads  = gen_tape.gradient(gen_total_loss, generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss,     discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gen_grads,  generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return gen_gan_loss, gen_total_loss, disc_loss

## 10. Training loop (leave-one-out cross-validation)

`fit` trains on every subject except one held-out test subject, and tracks the spatial correlation
and MSE between the generated and empirical task FC of that test subject every 100 steps. An
optional learning-rate schedule / decay is supported.

In [ ]:
def fit(input_data, output_data, steps, test_index,
        initial_lr=2e-4, lr_decay_factor=None, lr_schedule=None, batch_size=1):

    # --- leave-one-out split ---
    test_in  = tf.convert_to_tensor(input_data[test_index][None, ...],  tf.float32)
    test_out = tf.convert_to_tensor(output_data[test_index][None, ...], tf.float32)

    train_idx = [i for i in range(len(input_data)) if i != test_index]
    train_in  = tf.convert_to_tensor(input_data[train_idx],  tf.float32)
    train_out = tf.convert_to_tensor(output_data[train_idx], tf.float32)

    ds = (tf.data.Dataset.from_tensor_slices((train_in, train_out))
          .shuffle(len(train_idx)).repeat().batch(batch_size))

    generator_optimizer.learning_rate.assign(initial_lr)
    discriminator_optimizer.learning_rate.assign(initial_lr)
    print(f"Subjects: {len(input_data)} | test idx: {test_index} | "
          f"train: {len(train_idx)} | initial lr: {initial_lr:.6f}")

    history = {'corr': [], 'gen_total': [], 'gen_gan': [], 'disc': []}
    t0 = tick = time.time()

    for step, (inp, tar) in enumerate(ds.take(steps)):
        # --- periodic learning-rate adjustment ---
        if step % 1000 == 0:
            if step != 0:
                print(f"  [{step//1000}k/{steps//1000}k] last 1000 steps: {time.time()-tick:.1f}s")
                lr = generator_optimizer.learning_rate.numpy()
                if lr_schedule and (step // 1000) < len(lr_schedule):
                    lr = lr_schedule[step // 1000]
                elif lr_decay_factor is not None:
                    lr = lr * lr_decay_factor
                generator_optimizer.learning_rate.assign(lr)
                discriminator_optimizer.learning_rate.assign(lr)
            tick = time.time()

        gen_gan, gen_total, disc = train_step(inp, tar)

        # --- evaluation on the held-out subject ---
        if (step + 1) % 100 == 0:
            pred = generator(test_in, training=False)
            corr = float(spatial_correlation_corr2(test_out, pred))
            mse  = calculate_mse(pred.numpy(), test_out.numpy())
            history['corr'].append(corr)
            history['gen_total'].append(float(gen_total))
            history['gen_gan'].append(float(gen_gan))
            history['disc'].append(float(disc))
            print(f"  step {step+1:5d} | gen_total {float(gen_total):.4f} "
                  f"gen_gan {float(gen_gan):.4f} disc {float(disc):.4f} "
                  f"| test corr {corr:.4f}  mse {mse:.5f}")

    print(f"Done (test idx {test_index}). Total time: {time.time()-t0:.1f}s")
    return history

## 11. Run training

Train on all subjects except subject `TEST_INDEX`, then evaluate on the held-out subject.
Loop over `range(N_SUBJECTS)` to reproduce full leave-one-out cross-validation.

In [ ]:
STEPS      = 20000
TEST_INDEX = 0

history = fit(input_data, output_data,
              steps=STEPS, test_index=TEST_INDEX,
              initial_lr=2e-4)